# 08 · Evaluation: precision, recall, and mAP

**Level:** Intermediate → advanced · **Time:** 40–55 minutes.
**Prerequisites:** bounding boxes and IoU; notebook 07 only for the optional
real-model section. The core lesson runs offline with NumPy and Matplotlib.

**You will:** match detections to ground truth one-to-one, investigate duplicate
and missed detections, sweep confidence, and compute educational 101-point AP.

**Demo story:** “Six predicted boxes for four objects: is that good?” Counts
alone cannot tell us. We need class labels, box overlap, confidence, and a rule
that prevents two predictions from claiming the same object.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the ObjectDetection project folder (see README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, show_boxes
seed_everything(42)
print("Project:", ROOT)


In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Two deliberately simple images. These are hand-written examples, not model output.
ground_truth = [
    {"image": "A", "class": 0, "box": [10, 10, 50, 50]},
    {"image": "A", "class": 1, "box": [65, 10, 100, 50]},
    {"image": "B", "class": 0, "box": [20, 20, 60, 60]},
    {"image": "B", "class": 1, "box": [70, 70, 110, 110]},
]
predictions = [
    {"image": "A", "class": 0, "box": [10, 10, 50, 50], "score": .95, "note": "correct"},
    {"image": "A", "class": 0, "box": [11, 11, 50, 50], "score": .90, "note": "duplicate"},
    {"image": "A", "class": 1, "box": [66, 10, 101, 50], "score": .85, "note": "correct"},
    {"image": "B", "class": 0, "box": [23, 23, 63, 63], "score": .80, "note": "slightly shifted"},
    {"image": "B", "class": 1, "box": [0, 80, 20, 100], "score": .70, "note": "background"},
    {"image": "A", "class": 1, "box": [10, 10, 50, 50], "score": .60, "note": "wrong class"},
]
fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for row, image_id in enumerate(["A", "B"]):
    canvas = np.full((120, 120, 3), 245, dtype=np.uint8)
    gt = [g for g in ground_truth if g["image"] == image_id]
    pred = [p for p in predictions if p["image"] == image_id]
    show_boxes(canvas, [g["box"] for g in gt], [g["class"] for g in gt],
               ax=axes[row, 0], title=f"{image_id}: ground truth")
    show_boxes(canvas, [p["box"] for p in pred], [p["class"] for p in pred],
               [p["score"] for p in pred], ax=axes[row, 1], title=f"{image_id}: toy predictions")
plt.tight_layout()
plt.show()
display(pd.DataFrame(predictions))


## 1. Match boxes one-to-one

Sort predictions by confidence, highest first. For each prediction, consider
**unmatched ground-truth boxes in the same image and class**. The highest-IoU
eligible box is a match if IoU reaches the chosen threshold. A match is a true
positive (TP); an unmatched prediction is a false positive (FP). Unmatched
ground truth becomes a false negative (FN).

**Presenter prompt:** Before running, count how many predictions should be true
positives at IoU 0.50. Explain why the duplicate must count as a false positive.

Coordinates here use continuous `xyxy`: area is `(x2-x1)*(y2-y1)` without `+1`.


In [ ]:
def iou_one_to_many(box, boxes):
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)
    if len(boxes) == 0:
        return np.zeros(0)
    box = np.asarray(box, dtype=float)
    top_left = np.maximum(box[:2], boxes[:, :2])
    bottom_right = np.minimum(box[2:], boxes[:, 2:])
    intersection = np.prod(np.maximum(bottom_right - top_left, 0), axis=1)
    area = np.prod(np.maximum(box[2:] - box[:2], 0))
    other_area = np.prod(np.maximum(boxes[:, 2:] - boxes[:, :2], 0), axis=1)
    return intersection / np.maximum(area + other_area - intersection, 1e-12)

def match_detections(preds, truth, iou_threshold=.50, confidence=0.0):
    remaining = set(range(len(truth)))
    rows = []
    for p in sorted((p for p in preds if p["score"] >= confidence),
                    key=lambda p: p["score"], reverse=True):
        candidates = [j for j in sorted(remaining)
                      if truth[j]["image"] == p["image"] and truth[j]["class"] == p["class"]]
        overlaps = iou_one_to_many(p["box"], [truth[j]["box"] for j in candidates])
        best = int(np.argmax(overlaps)) if len(overlaps) else None
        is_tp = best is not None and overlaps[best] >= iou_threshold
        if is_tp:
            remaining.remove(candidates[best])
        rows.append({**p, "TP": int(is_tp), "FP": int(not is_tp),
                     "matched_iou": float(overlaps[best]) if is_tp else 0.0})
    return rows, [truth[j] for j in sorted(remaining)]

matched, missed = match_detections(predictions, ground_truth)
display(pd.DataFrame(matched)[["image", "class", "score", "note", "TP", "FP", "matched_iou"]])
print("Missed ground-truth objects:", missed)
tp, fp, fn = sum(r["TP"] for r in matched), sum(r["FP"] for r in matched), len(missed)
assert (tp, fp, fn) == (3, 3, 1)
assert next(r for r in matched if r["note"] == "duplicate")["FP"] == 1
print(f"TP={tp}; FP={fp}; FN={fn}")


## 2. Confidence is an operating decision

**Precision = TP / (TP + FP)** asks how many reported objects are correct.
**Recall = TP / (TP + FN)** asks how many real objects were found.
**F1** is their harmonic mean. A confidence threshold discards predictions;
an IoU threshold decides whether a retained box is localized well enough.
These are different controls.

Here we report precision as zero when no predictions remain. Other reporting
conventions may mark it undefined; always state the convention.


In [ ]:
sweep_rows = []
for threshold in [0.0, .65, .75, .82, .88, .93, .99]:
    rows, remaining = match_detections(predictions, ground_truth, confidence=threshold)
    tp = sum(r["TP"] for r in rows)
    fp = sum(r["FP"] for r in rows)
    fn = len(remaining)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    sweep_rows.append(dict(confidence=threshold, TP=tp, FP=fp, FN=fn,
                           precision=precision, recall=recall, F1=f1))
sweep = pd.DataFrame(sweep_rows)
display(sweep.round(3))
sweep.plot(x="confidence", y=["precision", "recall", "F1"], marker="o", ylim=(0, 1.05),
           figsize=(8, 4), title="Toy detections: threshold trade-off at IoU 0.50")
plt.show()


## 3. Average precision summarizes a ranked list

For each class, accumulate TP/FP along the confidence-sorted predictions to
obtain precision and recall. At each recall level from 0.00 to 1.00 in steps
of 0.01, take the best precision achieved at that recall or higher; use zero
for an unreachable recall level. The mean of those 101 values is **AP101**.

Average AP across classes to get mAP for one IoU threshold. Average again over
IoU 0.50, 0.55, …, 0.95 for the educational mAP50–95 below. Missing classes with
ground truth get AP=0; a class with no ground truth is undefined and excluded.

This is a small transparent evaluator, **not the complete COCO evaluator**:
it omits crowd/ignore annotations, area ranges, maximum-detection limits and
other benchmark details. Use the official/established evaluator for a benchmark.


In [ ]:
def class_ap101(preds, truth, class_id, iou_threshold=.50):
    class_gt = [g for g in truth if g["class"] == class_id]
    if not class_gt:
        return np.nan, np.array([]), np.array([])
    class_preds = [p for p in preds if p["class"] == class_id]
    rows, _ = match_detections(class_preds, class_gt, iou_threshold=iou_threshold)
    cumulative_tp = np.cumsum([r["TP"] for r in rows], dtype=float)
    cumulative_fp = np.cumsum([r["FP"] for r in rows], dtype=float)
    recall = cumulative_tp / len(class_gt)
    precision = cumulative_tp / np.maximum(cumulative_tp + cumulative_fp, 1)
    recall_grid = np.linspace(0, 1, 101)
    interpolated = [precision[recall >= r].max() if np.any(recall >= r) else 0.0
                    for r in recall_grid]
    return float(np.mean(interpolated)), precision, recall

classes = sorted({g["class"] for g in ground_truth})
iou_thresholds = np.linspace(.50, .95, 10)
ap_rows = []
for iou_threshold in iou_thresholds:
    for class_id in classes:
        ap, _, _ = class_ap101(predictions, ground_truth, class_id, iou_threshold)
        ap_rows.append({"IoU": round(float(iou_threshold), 2), "class": class_id, "AP101": ap})
ap_table = pd.DataFrame(ap_rows)
display(ap_table.pivot(index="IoU", columns="class", values="AP101").round(3))
print("Educational mAP50:", round(ap_table.loc[ap_table["IoU"] == .5, "AP101"].mean(), 4))
print("Educational mAP50–95:", round(ap_table["AP101"].mean(), 4))
# A perfect set scores 1; no predictions score 0 when ground truth exists.
perfect = [{**g, "score": 1.0} for g in ground_truth]
assert np.isclose(class_ap101(perfect, ground_truth, 0)[0], 1.0)
assert class_ap101([], ground_truth, 0)[0] == 0.0


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for class_id in classes:
    ap, precision, recall = class_ap101(predictions, ground_truth, class_id)
    axes[0].step(recall, precision, where="post", marker="o", label=f"class {class_id}: AP={ap:.3f}")
axes[0].set(xlim=(0, 1.05), ylim=(0, 1.05), xlabel="Recall", ylabel="Precision",
            title="Raw ranked precision/recall points")
axes[0].legend()
ap_table.groupby("IoU")["AP101"].mean().plot(ax=axes[1], marker="o")
axes[1].set(ylim=(0, 1.05), ylabel="Mean AP101", title="Stricter localization requirements")
plt.tight_layout()
plt.show()


## 4. Optional: evaluate the actual custom checkpoint

The next cell uses the local checkpoint manifest from notebook 07. It evaluates
the validation split with Ultralytics' implementation, allowing very low
confidence predictions into the ranked evaluation (`conf=0.001`). That is
different from selecting a threshold for a live demo.

Compare the model's own metrics with visual failure cases. Do not compare these
scores directly to the hand-written examples above: their data and evaluator
details differ. Keep `split="val"` while tuning; evaluate the reserved test split
once after choosing your configuration.


In [ ]:
model_metrics = None
manifest_path = ROOT / "artifacts" / "custom_model.json"
OFFLINE = os.environ.get("COURSE_OFFLINE", "0") == "1"
if OFFLINE:
    print("Offline lesson mode: optional Ultralytics validation skipped.")
elif not manifest_path.exists():
    print("No checkpoint manifest. Complete notebook 07 to run real-model validation.")
else:
    from ultralytics import YOLO
    from course_utils import choose_device
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    weights = Path(manifest["best_weights"])
    if not weights.exists():
        print("Manifest points to a moved/deleted checkpoint. Rerun notebook 07:", weights)
    else:
        model = YOLO(str(weights))
        model_metrics = model.val(data=manifest["data_yaml"], split="val", imgsz=256,
                                  batch=8, workers=0, device=choose_device(), conf=.001,
                                  plots=True, project=str(ROOT / "artifacts" / "evaluation"),
                                  name="custom_validation", verbose=False)
        display(pd.DataFrame([model_metrics.results_dict]))
        print("Validation plots:", model_metrics.save_dir)


## 5. Exercises and demonstration checkpoints

1. Remove the duplicate prediction. Does recall change? Does AP change?
2. Add a correct class-1 box for image B at confidence 0.55. What happens to
   achievable recall, even though a higher deployment threshold may hide it?
3. Change the IoU threshold to 0.75. Which previous TP becomes an FP, and why
   does a corresponding FN appear?
4. Choose a threshold for a task where missed objects are more costly than
   extra alerts. Explain why mAP alone does not choose that threshold.

<details><summary>Solutions</summary>

1. Recall stays the same; removing a high-ranked FP increases class-0 AP.
2. The class can reach recall 1.0. AP improves, but a threshold above 0.55 still
   hides that box. Confidence is not a calibrated probability by default.
3. The shifted class-0 box in B has IoU about 0.748 and fails at 0.75. Its
   prediction is an FP and the unmatched ground-truth object is an FN.
4. Usually start with a lower threshold, then measure the alert burden on
   representative validation data. The acceptable trade-off is application-specific.
</details>

**Checkpoint:** Explain why a detection can be correctly classified but fail
localization, and why two boxes on one object cannot both be TPs.

**Primary references:** [COCO detection evaluation](https://cocodataset.org/#detection-eval),
[COCO evaluator implementation](https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/cocoeval.py),
[Ultralytics validation](https://docs.ultralytics.com/modes/val/).
